# 04 — 第三代：Hybrid Pipeline + Data Recovery

**方法论定位（第三代，Nemotron-CC 2024）**

三个核心创新：
1. **分类器集成（Classifier Ensembling）**：多个分类器取并集，扩大高质量覆盖面
2. **条件性 Heuristic Bypass**：对高质量文档跳过 heuristic，减少误杀（Nemotron-CC 发现误杀率 18.1%）
3. **合成数据改写（Synthetic Rephrasing）**：低质量数据用 LLM API 改写后回收

**与第二代的核心区别**：
- 第二代丢弃 90% 数据（激进过滤）
- 第三代在质量不降的前提下保留约 38% 数据（4倍于第二代）
- Nemotron-CC：8B 模型 15T token 训练后超过 Llama 3.1 8B：MMPU +5, ARC-Challenge +3.1

## Cell Group A: 分类器集成（Classifier Ensembling）

> **为什么需要集成？单一分类器的盲区问题**
>
> 无论正样本是 OpenHermes（DCLM 风格）还是 Wikipedia（教育风格），
> 单一分类器都会有覆盖盲区。
> 例如：技术博客可能被“OpenHermes 风格”分类器低估，
> 却被“教育类”分类器高估。
>
> **Union 策略**：任一分类器认为高质量 → 判为高质量
> - 优点：扩大覆盖面，减少漏网之鱼
> - 缺点：可能引入更多噪声（对比 Intersection 策略）
>
> **Intersection 策略**：所有分类器都认为高质量
> - 类似第二代，更保守，质量更高但量更少
>
> Nemotron-CC 使用 Union 策略，实现了 +28% unique token 覆盖。

In [ ]:
import sys, json, random
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.utils.config_loader import load_run_config, load_pipeline_config, load_eval_config, print_config_summary
from src.gen2.quality_classifier import Gen2QualityClassifier
from src.gen3.classifier_ensemble import ClassifierEnsemble

run_cfg = load_run_config()
print_config_summary(run_cfg)

# 加载数据
from src.gen1.pipeline import read_jsonl
gen1_output = Path('../data/gen1_output/gen1_output.jsonl')
raw_files = list(Path('../data/raw').glob('*.jsonl'))

if gen1_output.exists():
    docs = read_jsonl(gen1_output, doc_limit=run_cfg['doc_limit'])
    print(f"✅ 读取 Gen1 输出: {len(docs):,} 条")
else:
    # 模拟数据
    docs = [{'text': f'Document {i}: ' + ('high quality scientific content about machine learning and AI. ' if i % 3 == 0 else 'buy cheap products now discount sale click here. ' if i % 3 == 1 else 'random web page content with mixed quality. ') * 15,
             'url': f'http://example{i}.com'}
            for i in range(run_cfg['doc_limit'])]
    print(f"✅ 使用模拟数据: {len(docs):,} 条")

texts = [d['text'] for d in docs]

In [ ]:
# 构建分类器集成
ensemble = ClassifierEnsemble(strategy='union', union_threshold=0.5)

# 分类器 1: DCLM 风格 (复用 Gen2)
clf1_path = '../results/quality_scores/gen2_classifier.bin'
if Path(clf1_path).exists():
    clf1 = Gen2QualityClassifier(model_path=clf1_path)
    ensemble.add_fasttext_classifier('fasttext_dclm', clf1, weight=0.4)
    print("✅ 注册 fasttext_dclm 分类器")
else:
    print(f"⚠️  Gen2 分类器不存在，请先运行 Notebook 03 或 run_gen2.py")

# 分类器 2: TF-IDF + LR（使用 Wikipedia 正样本）
wiki_texts = []
wiki_path = Path('../data/reference/wikipedia_abstracts.jsonl')
if wiki_path.exists():
    with open(wiki_path) as f:
        for i, line in enumerate(f):
            if i >= 1000: break
            try: wiki_texts.append(json.loads(line)['text'])
            except: pass

if not wiki_texts:
    wiki_texts = ['Scientific knowledge about physics, chemistry, and biology forms the basis of modern science.' * 3] * 500

neg_sample = [d['text'] for d in docs[:len(wiki_texts)]]
ensemble.train_tfidf_lr(
    name='tfidf_lr_wiki',
    positive_texts=wiki_texts,
    negative_texts=neg_sample,
    model_path='../results/quality_scores/gen3_tfidf_lr.pkl',
    weight=0.3,
)

In [ ]:
# 对全量文档打分并比较覆盖率
print("集成打分中...")
ensemble_scores, individual_scores = ensemble.score_batch(texts)

# 覆盖率对比
coverage = ensemble.compare_coverage(texts, threshold=0.5)
print(f"\n📊 分类器覆盖率对比（Union 策略）:")
for name, stats in coverage.get('individual_coverage', {}).items():
    print(f"  {name}: {stats['count']:,} 条 ({stats['rate']:.1%})")

ens = coverage.get('ensemble_coverage', {})
print(f"  集成(Union): {ens.get('count', 0):,} 条 ({ens.get('rate', 0):.1%})")

union_vs_best = coverage.get('union_vs_best_single', {})
if union_vs_best:
    print(f"\n  Union vs 最佳单分类器:")
    print(f"  额外覆盖: +{union_vs_best.get('gain', 0):,} 条 文档")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, scores_arr in individual_scores.items():
    axes[0].hist(scores_arr, bins=30, alpha=0.5, label=name)
axes[0].hist(ensemble_scores, bins=30, alpha=0.4, label='集成分数', color='black')
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('文档数')
axes[0].set_title('各分类器分数分布对比')
axes[0].legend(fontsize=8)

# 覆盖率柱状图
clf_names = list(individual_scores.keys()) + ['集成(Union)']
clf_rates = [np.mean(s >= 0.5) for s in individual_scores.values()] + [np.mean(ensemble_scores >= 0.5)]
bars = axes[1].bar(clf_names, [r*100 for r in clf_rates],
                    color=['#FF9800', '#17a2b8', '#28a745'], alpha=0.85)
axes[1].set_ylabel('高质量文档覆盖率 (%)')
axes[1].set_title('不同策略的覆盖率对比')
axes[1].set_xticklabels(clf_names, rotation=15)
for bar, rate in zip(bars, clf_rates):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{rate:.1%}', ha='center', va='bottom', fontsize=10)
plt.suptitle('第三代：分类器集成分析', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../results/figures/04_gen3_ensemble_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell Group B: 条件性 Heuristic Bypass

> **核心问题：Heuristic 会误杀多少高质量文档？**
>
> Nemotron-CC 的关键发现：对 fastText 判定为高质量的文档，
> 如果再应用 heuristic filter，会误杀 **18.1% 的高质量 token**。
>
> **误杀的原因举例**：
> - 代码文档：含大量特殊字符 → 被 Gopher 的“alpha ratio”规则过滤
> - 技术教程：含代码片段（短行）→ 被 C4 的行规则过滤
> - 问答格式文本：平均句子短 → 被 Gopher 的 avg_sentence_length 过滤
>
> **解决方案（Bypass 路由）**：
> - score >= 0.7：直接保留（跳过 heuristic）
> - 0.3 <= score < 0.7：应用 heuristic
> - score < 0.3：送去 LLM 改写或丢弃

In [ ]:
from src.gen3.conditional_bypass import ConditionalBypass
from src.gen1.filters.quality_filter import QualityFilter

router = ConditionalBypass(
    high_quality_threshold=0.7,
    medium_quality_threshold=0.3,
    rephrase_score_range=(0.1, 0.3),
)
quality_filter = QualityFilter()

# 执行路由
buckets = router.route(docs, ensemble_scores, quality_filter)

# Bypass 价值分析（验证 18.1% 发现）
bypass_analysis = router.compute_bypass_value(
    buckets['high_quality'], quality_filter
)

print(f"\nBypass 价值验证（对标 Nemotron-CC 18.1%）:")
print(f"  本实验误杀率: {bypass_analysis.get('would_be_filtered_rate', 0):.1%}")
print(f"  Nemotron-CC 论文: 18.1%")

# 路由漏斗图
summary = router.get_summary(buckets, len(docs))
fig, ax = plt.subplots(figsize=(10, 5))
route_names = ['高质量\n(直接保留)', '中等->Heuristic\n通过', '中等->Heuristic\n过滤', '待改写', '丢弃']
route_counts = [
    summary.get('high_quality', {}).get('count', 0),
    summary.get('heuristic_passed', {}).get('count', 0),
    summary.get('heuristic_filtered', {}).get('count', 0),
    summary.get('to_rephrase', {}).get('count', 0),
    summary.get('discarded', {}).get('count', 0),
]
route_colors = ['#28a745', '#4CAF50', '#FF9800', '#17a2b8', '#6c757d']
bars = ax.bar(route_names, route_counts, color=route_colors, alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, route_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(route_counts)*0.01,
            f'{count:,}\n({count/len(docs):.1%})', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('文档数')
ax.set_title('第三代：条件性路由结果（Bypass 分析）', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/04_gen3_routing.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell Group C: 合成数据改写（Synthetic Rephrasing）

> **核心理念：低质量数据不是垃圾，而是待改写的原材料**
>
> Nemotron-CC 的洞见：通过 LLM API 将低质量文本改写为高质量文本，实现数据回收。
>
> **成本效益分析**：
> - Anthropic Haiku：约 $0.0003/千 token（输入）
> - 改写 300 条文档（full_run 配置）≈ 300 x 500 tokens = 150K tokens ≈ **$0.05**
> - 这 300 条变成高质量数据后，价值 >> 成本
>
> **改写后的质量验证**：
> 改写后的文档需要通过评估分类器验证（分数 >= 0.4 才保留），
> 避免“改写失败”（LLM 生成了更差的文本）的情况。

In [ ]:
# LLM 改写演示（需要 API Key）
try:
    from src.utils.config_loader import load_api_config
    from src.gen3.synthetic_rephraser import SyntheticRephraser

    api_cfg = load_api_config()
    rephraser = SyntheticRephraser(api_cfg)

    # 从 to_rephrase 桶中取样
    rephrase_candidates = buckets.get('to_rephrase', [])
    rewrite_count = min(run_cfg.get('rewrite_count', 50), len(rephrase_candidates))

    if rewrite_count > 0:
        print(f"\u270d\ufe0f \u6539\u5199 {rewrite_count} \u6761\u6587\u6863...")
        rephrased_docs, rephrase_stats = rephraser.rephrase_batch(
            rephrase_candidates, max_count=rewrite_count
        )
        print(f"\n\u6539\u5199\u7ed3\u679c:")
        for k, v in rephrase_stats.items():
            if k != 'status_counts':
                print(f"  {k}: {v}")

        # 改写前后对比
        if rephrased_docs:
            orig = rephrase_candidates[0]
            reph = rephrased_docs[0]
            print(f"\n\u6539\u5199\u6848\u4f8b\u5bf9\u6bd4:")
            print(f"  \u539f\u6587\uff08\u524d200\u5b57\uff09: {orig['text'][:200]!r}")
            print(f"  \u6539\u5199\u540e\uff08\u524d200\u5b57\uff09: {reph['text'][:200]!r}")
    else:
        print("\u26a0\ufe0f \u6ca1\u6709\u5f85\u6539\u5199\u7684\u6587\u6863")
        rephrased_docs = []

except ValueError as e:
    print(f"\u26a0\ufe0f  {e}")
    print("\u8df3\u8fc7 LLM \u6539\u5199\u6b65\u9aa4\uff08\u9700\u8981\u8bbe\u7f6e API Key\uff09")
    print("\u8bbe\u7f6e\u65b9\u6cd5\uff1a\u5728 configs/api_config.yaml \u4e2d\u586b\u5165 api_key")
    rephrased_docs = []

In [ ]:
# 第三代最终汇总
final_gen3 = (
    buckets.get('high_quality', []) +
    buckets.get('heuristic_passed', []) +
    rephrased_docs
)

total_input = len(docs)
total_output = len(final_gen3)

print("=" * 60)
print("  第三代 Hybrid Pipeline — 最终结论")
print("=" * 60)
print(f"  输入文档数: {total_input:,}")
print(f"  最终输出: {total_output:,} 条")
print(f"  \u251c\u2500\u2500 高质量(bypass): {len(buckets.get('high_quality', [])):,} 条")
print(f"  \u251c\u2500\u2500 中等(heuristic通过): {len(buckets.get('heuristic_passed', [])):,} 条")
print(f"  \u2514\u2500\u2500 合成数据(改写): {len(rephrased_docs):,} 条")
print(f"  总保留率: {total_output/total_input:.1%}")
print()
print("  对比第二代（top-10%）:")
gen2_count = int(total_input * 0.10)
print(f"  第二代输出约: {gen2_count:,} 条")
print(f"  第三代输出约: {total_output:,} 条")
if gen2_count > 0:
    print(f"  数据量倍数: {total_output/gen2_count:.1f}x")
print()
print("  下一步 -> Notebook 05：去重分析")